# vLLM inference benchmark
This notebook visualizes measured results only. Generate the CSV with `inference-lab/benchmark_vllm.py`; do not substitute theoretical FP8 savings for peak VRAM measurements.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

results = pd.read_csv(Path('../inference-lab/results/benchmark.csv'))
results.head()

In [ ]:
vram = results.groupby('quantization')['peak_vram_gib'].max().dropna()
if {'bf16', 'fp8'}.issubset(vram.index):
    saving = (vram['bf16'] - vram['fp8']) / vram['bf16'] * 100
    ax = vram.loc[['bf16', 'fp8']].plot.bar(color=['#64748b', '#14b8a6'])
    ax.set_title(f'Measured peak VRAM — FP8 saves {saving:.1f}%')
    ax.set_ylabel('GiB')
    plt.show()

In [ ]:
ttft = results.groupby(['requested_prompt_tokens', 'chunked_prefill']).ttft_ms.quantile(.95).unstack()
ttft.plot(marker='o', title='Chunked prefill — measured P95 TTFT', ylabel='ms')
plt.show()
throughput = results.groupby(['block_size', 'concurrency']).output_tokens_per_second.mean().unstack()
throughput.plot(marker='o', title='Decode throughput by block_size', ylabel='tokens/s')
plt.show()